In [93]:
#V-measure als Quantitative Maße für die Qualität deines ESM-Clusterings im Vergleich zu Canonical Clustern
#Werte liegen zwsichen 0 (schlechte überinstimmung) und 1 (perfekte Überinstimmung)

In [115]:

import os
import glob
import pandas as pd
from functools import reduce
from sklearn.metrics import v_measure_score


In [116]:
#input Datei filtern, für finalen datensatz
seq_regions = ['SEQ_H1', 'SEQ_H2', 'SEQ_L1', 'SEQ_L2', 'SEQ_L3']
cf_regions = ['CF_H1', 'CF_H2', 'CF_L1', 'CF_L2', 'CF_L3']
df = (
    pd.read_csv("data/ab_ag_scalop.tsv", sep="\t")
    .dropna(subset=seq_regions)
    .dropna(subset=cf_regions)
    .drop_duplicates(subset=seq_regions) 
)
antigen_counts = df["antigen_name"].value_counts() # Tabelle aus antigen_names und ihren Häufigkeiten in der Spalte antigen_name
df = df[df["antigen_name"].isin(antigen_counts[antigen_counts >= 5].index)] # Behält nur Zeilen, deren antigen_name mindestens 5-mal vorkommt
    

In [ ]:
#TSV dateien für die verschiedenen CDR-Regionen einlesen und zusammenführen in eine Datei die als input für V-measure dient
#Input datei wird in ESMC erstellt

# CDR-Regionen definieren für die benennung der Spalten
cdrs = ["H1", "H2", "L1", "L2", "L3"]

# Mapping zwischen CF-Spalten und neuen HC-Spalten
cf_columns = [f"CF_{cdr}" for cdr in cdrs]
hc_columns = [f"HC_{cdr}" for cdr in cdrs]

# Lege leere HC-Spalten an
for hc_col in hc_columns:
    df[hc_col] = None

# Füge pro CDR-Typ die Clusterlabels hinzu
for cdr in cdrs:
    cluster_df = pd.read_csv(f"data/cdr_cluster_ESMC_tsvs/clusters_SEQ_{cdr}.tsv", sep="\t")

    # Mapping: pdb_id → cluster_label
    cluster_map = dict(zip(cluster_df["pdb"], cluster_df["cluster"]))

    # Schreibe ins Haupt-DataFrame
    df[f"HC_{cdr}"] = df["pdb"].map(cluster_map)

# Speichern als neue Datei
os.makedirs("data", exist_ok=True)
df.to_csv("data/ab_ag_hierarchical_canonical_forms_ESMC.csv", index=False)

In [97]:
# v measure für ESMC clustering

# Dateien einlesen
df_true = df  # Original
df_pred = pd.read_csv("data/ab_ag_hierarchical_canonical_forms_ESMC.csv")  # Hierarchische Cluster

# CDRs, die verglichen werden sollen
cdrs = ["H1", "H2", "L1", "L2", "L3"]

print("V-Measure Ergebnisse:\n")

# Für jede Region CF vs HC vergleichen
for cdr in cdrs:
    true_labels = df_true[f"CF_{cdr}"]
    pred_labels = df_pred[f"HC_{cdr}"]
    
    v_score = v_measure_score(true_labels, pred_labels)
    print(f"CDR {cdr}: V-Measure = {v_score:.3f}")

V-Measure Ergebnisse:

CDR H1: V-Measure = 0.138
CDR H2: V-Measure = 0.233
CDR L1: V-Measure = 0.523
CDR L2: V-Measure = 0.000
CDR L3: V-Measure = 0.238


In [117]:
# einzelne Files der verschiedenen CDRs zusammenführen für ESMC clustering wo vorher nach länge sortiert wurde. 
# Daraus haben wir mehrere Dateien pro Region und führen die zusammen, dass nur noch eine pro region übrig bleibt.
#Input datei wird in ESMC erstellt

#Input- und Output-Pfade
#Input Pfad
in_dir  = "data/cdr_cluster_ESMC_by_length_tsvs" 
# Zielordner für die zusammengeführten TSVs
output_dir = "data/merged_clusters_per_region" 
# Erstelle output_dir, falls es noch nicht existiert
os.makedirs(output_dir, exist_ok=True) 

# Liste der CDR-Regionen, für die wir Dateien brauchen
regions = ["H1", "H2", "L1", "L2", "L3"]
#Pro Region alle Dateien einlesen und mergen
for region in regions:
    #dateinamen erstellen
    pattern = os.path.join(in_dir, f"clusters_SEQ_{region}_len*.tsv")
    files = glob.glob(pattern)
    if not files:
        print(f" Keine Dateien für SEQ_{region} gefunden ({pattern})")
        continue

    # Leere Liste, um jeweils den DataFrame einer Datei zwischenzuspeichern  
    dfs = []
    # Jede gefundene Datei verarbeiten
    for fn in files:
        #pdb ID eingelesen und als string übernehmen
        df_length = pd.read_csv(fn, sep="\t", dtype={"pdb": str})
        for c in ("Hchain", "Lchain"):
            if c not in df_length.columns:
                df_length[c] = ""
            
       #definieren der neuen spalte die region, länge der Sequenz und Cluster-ID enthält
        col_new = f"HC_{region}"
        df_length[col_new] = (
            region
            + "_" + df_length["length"].astype(str) 
            + "_" + df_length["cluster"].astype(str)
        )

        # nur die vier Spalten behalten
        dfs.append(df_length[["pdb","Hchain","Lchain",col_new]])

    # alle Teildateien zusammenkleben
    merged = pd.concat(dfs, ignore_index=True)

    # abspeichern
    out_fn = os.path.join(output_dir, f"merged_clusters_SEQ_{region}.tsv")
    merged.to_csv(out_fn, sep="\t", index=False)
    

In [118]:
#erstellen der Input Datei für V-measure für ESMC clustering nach länge

# Input Ordner mit Dateien wo die verschiedenen Dateien für einen CDR zusammengeführt wurden
in_dir  = "data/merged_clusters_per_region"
# Zielordner für die zusammengeführten TSVs
out_fn  = "data/ab_ag_clustered_all_CDRs_ESMC_by_length.csv"
# Erstelle output_dir, falls es noch nicht existiert
os.makedirs(os.path.dirname(out_fn), exist_ok=True)

# Liste der CDR-Regionen, für die wir Dateien brauchen
regions = ["H1","H2","L1","L2","L3"]

# Liste für alle DataFrames, einen pro Region 
dfs = []
for r in regions:
    # Dateinamen erstellen
    fn = os.path.join(in_dir, f"merged_clusters_SEQ_{r}.tsv")
    #alle Spalten als strings
    df_length = pd.read_csv(fn, sep="\t", dtype=str)
    dfs.append(df_length)

# Alle auf pdb+Hchain+Lchain mergen (outer, damit nichts verloren geht) und 
merged = reduce(
    lambda left, right: left.merge(
        right,
        on=["pdb","Hchain","Lchain"],  # Gemeinsame Schlüsselspalten für den Join
        how="outer" # behalten aller Zeilen beider DataFrames
    ),
    dfs
)

# fehlendes auffüllen und Spalten anordnen
merged.fillna("", inplace=True)
cols = ["pdb","Hchain","Lchain"] + [f"HC_{r}" for r in regions]
merged = merged[cols]

# Ergebnis speichern
merged.to_csv(out_fn, index=False)

In [119]:
# v measure ESMC nach Länge vorsortiert

# Dateien einlesen
df_true = df  # Original
df_pred = pd.read_csv("data/ab_ag_clustered_all_CDRs_ESMC_by_length.csv")  # Hierarchische Cluster

print(df_pred.columns.tolist())
# CDRs, die verglichen werden sollen
cdr = ["H1", "H2", "L1", "L2", "L3"]

print("V-Measure Ergebnisse:\n")

# Für jede Region CF vs HC vergleichen
for cdr in cdrs:
    true_labels = df_true[f"CF_{cdr}"]
    pred_labels = df_pred[f"HC_{cdr}"]
    
    v_score = v_measure_score(true_labels, pred_labels)
    print(f"CDR {cdr}: V-Measure = {v_score:.3f}")

['pdb', 'Hchain', 'Lchain', 'HC_H1', 'HC_H2', 'HC_L1', 'HC_L2', 'HC_L3']
V-Measure Ergebnisse:

CDR H1: V-Measure = 0.028
CDR H2: V-Measure = 0.033
CDR L1: V-Measure = 0.085
CDR L2: V-Measure = 0.000
CDR L3: V-Measure = 0.044


In [111]:
#erstellen der Input Datei für V-measure für ESMC clustering nach Länge mit vorbestimmter anzahl an Clustern basierend auf den referenzclustern
#Daraus haben wir mehrere Dateien pro Region und führen die zusammen, dass nur noch eine pro region übrig bleibt
#Input datei wird in ESMC erstellt

# Input Ordner mit Dateien wo die verschiedenen Dateien für einen CDR zusammengeführt wurden
in_dir  = "data/cdr_Cluster_ESMC_by_length_pre_defined_tsvs"
# Zielordner für die zusammengeführten TSVs
output_dir = "data/merged_clusters_per_region_pre defined"
# Erstelle output_dir, falls es noch nicht existiert
os.makedirs(output_dir, exist_ok=True)

# Liste der CDR-Regionen, für die wir Dateien brauchen
regions = ["H1","H2","L1","L2","L3"]

#Pro Region alle Dateien einlesen und mergen
for region in regions:
    #dateinamen erstellen
    pattern = os.path.join(in_dir, f"ESMC_clusters_SEQ_{region}_len*_pre_decided.tsv")
    files = glob.glob(pattern)
    if not files:
        print(f" Keine Dateien für SEQ_{region} gefunden ({pattern})")
        continue

    # Leere Liste, um jeweils den DataFrame einer Datei zwischenzuspeichern
    dfs = []

    # Jede gefundene Datei verarbeiten
    for fn in files:
        #pdb ID eingelesen und als string übernehmen
        df_length_pre = pd.read_csv(fn, sep="\t", dtype={"pdb": str})
        #stell sicher, dass Hchain/Lchain existieren
        for c in ("Hchain","Lchain"):
            if c not in df_length_pre.columns:
                df_length_pre[c] = ''

        #definieren der neuen spalte die region, länge der Sequenz und Cluster-ID enthält
        col_new = f"HC_{region}"
        df_length_pre[col_new] = (
            region
            + "_" + df_length_pre["Length"].astype(str)
            + "_" + df_length_pre["Cluster_Label"].astype(str)
        )

        # nur die vier Spalten behalten
        dfs.append(df_length_pre[["PDB_ID","Hchain","Lchain",col_new]])

    # alle Teildateien zusammenkleben
    merged = pd.concat(dfs, ignore_index=True)

    # abspeichern
    out_fn = os.path.join(output_dir, f"merged_clusters_SEQ_{region}.tsv")
    merged.to_csv(out_fn, sep="\t", index=False)

In [113]:
#erstellen der Input Datei für V-measure für ESMC clustering nach länge mit vorbestimmter anzahl an Clustern basierend auf den referenzclustern
#Daraus haben wir mehrere Dateien pro Region und führen die zusammen, dass nur noch eine pro region übrig bleibt

# Input Ordner mit Dateien wo die verschiedenen Dateien für einen CDR zusammengeführt wurden
in_dir = "data/merged_clusters_per_region_pre defined"
# Zielordner für die zusammengeführten TSVs
out_fn = "data/ab_ag_clustered_all_CDRS_ESMC_by_length_pre_defined.csv"
# Erstelle output_dir, falls es noch nicht existiert
os.makedirs(os.path.dirname(out_fn), exist_ok=True)

# Liste der CDR-Regionen, für die wir Dateien brauchen
regions = ["H1","H2","L1","L2","L3"]

# Liste für alle DataFrames, einen pro Region
dfs = []
for r in regions:
    # Dateinamen erstellen
    fn = os.path.join(in_dir, f"merged_clusters_SEQ_{r}.tsv")
    # alle Spalten als strings
    df_length_pre = pd.read_csv(fn, sep="\t", dtype=str)
    dfs.append(df_length_pre)

# Alle auf pdb+Hchain+Lchain mergen (outer, damit nichts verloren geht) und
merged = reduce(
    lambda left, right: left.merge(
        right,
        on=["PDB_ID","Hchain","Lchain"],  # Gemeinsame Schlüsselspalten für den Join
        how="outer"                    # behalten aller Zeilen beider DataFrames
    ),
    dfs
)

# fehlendes auffüllen und Spalten anordnen
merged.fillna("", inplace=True)
cols = ["PDB_ID","Hchain","Lchain"] + [f"HC_{r}" for r in regions]
merged = merged[cols]

# Ergebnis speichern
merged.to_csv(out_fn, index=False)


In [114]:
# v measure nach länge mit vorbestimmter anzahl an Clustern basierend auf den referenzclustern


# Dateien einlesen
df_true = df  # Original
df_pred = pd.read_csv("data/ab_ag_clustered_all_CDRs_ESMC_by_length_pre_defined.csv")  # Hierarchische Cluster

print(df_pred.columns.tolist())
# CDRs, die verglichen werden sollen
cdr = ["H1", "H2", "L1", "L2", "L3"]

print("V-Measure Ergebnisse:\n")

# Für jede Region CF vs HC vergleichen
for cdr in cdrs:
    true_labels = df_true[f"CF_{cdr}"]
    pred_labels = df_pred[f"HC_{cdr}"]
    
    v_score = v_measure_score(true_labels, pred_labels)
    print(f"CDR {cdr}: V-Measure = {v_score:.3f}")

['PDB_ID', 'Hchain', 'Lchain', 'HC_H1', 'HC_H2', 'HC_L1', 'HC_L2', 'HC_L3']
V-Measure Ergebnisse:

CDR H1: V-Measure = 0.022
CDR H2: V-Measure = 0.020
CDR L1: V-Measure = 0.041
CDR L2: V-Measure = 1.000
CDR L3: V-Measure = 0.015
